# **_[DELTA LIVE TABLES](url)_**

## **_[Bronze Layer](url)_**

#### **_[Customer Raw](url)_**

In [0]:
# Import Spark Declarative Pipelines module
from pyspark import pipelines as dp

# Define a streaming table in the bronze layer
@dp.table(
    name="databricks_demo.bronze.tb_customers_raw",  # Fully qualified table name in Unity Catalog
    comment="The customers lookup table, ingested from customer.csv file"  # Table description
)
def customers_raw():
    # Configure Auto Loader to incrementally ingest CSV files
    df = (
        spark.readStream.format("cloudFiles")  # Use Auto Loader for incremental file ingestion
        .option("cloudFiles.format", "csv")  # Specify source file format as CSV
        .option("cloudFiles.inferColumnTypes", "true")  # Automatically infer column data types
        .option("pathGlobFilter", "customers.csv")  # Filter to only process customers.csv file
        .load("/Volumes/databricks_demo/default/files_data/transaction_data/")  # Source directory path
    )
    return df  # Return streaming DataFrame for pipeline processing

#### **_[Product Raw](url)_**

In [0]:
# Import Spark Declarative Pipelines module
from pyspark import pipelines as dp

# Define a streaming table in the bronze layer for products data
@dp.table(
    name="databricks_demo.bronze.tb_products_raw",  # Fully qualified table name in Unity Catalog
    comment="The products lookup table, ingested from products.csv file"  # Table description
)
def products_raw():
    # Configure Auto Loader to incrementally ingest CSV files
    return (
        spark.readStream.format("cloudFiles")  # Use Auto Loader for incremental file ingestion
        .option("cloudFiles.format", "csv")  # Specify source file format as CSV
        .option("cloudFiles.inferColumnTypes", "true")  # Automatically infer column data types
        .option("pathGlobFilter", "products.csv")  # Filter to only process products.csv file
        .load("/Volumes/databricks_demo/default/files_data/transaction_data/")  # Source directory path
    )

#### **_[Orders Raw](url)_**

In [0]:
# Import Spark Declarative Pipelines module
from pyspark import pipelines as dp

# Define a streaming table in the bronze layer for orders data
@dp.table(
    name="databricks_demo.bronze.tb_orders_raw",  # Fully qualified table name in Unity Catalog
    comment="The orders table. ingested from orders.csv file"  # Table description
)
def orders_raw():
    # Configure Auto Loader to incrementally ingest CSV files
    return (
        spark.readStream.format("cloudFiles")  # Use Auto Loader for incremental file ingestion
        .option("cloudFiles.format", "csv")  # Specify source file format as CSV
        .option("cloudFiles.inferColumnTypes", "true")  # Automatically infer column data types
        .option("pathGlobFilter", "orders.csv")  # Filter to only process orders.csv file
        .load("/Volumes/databricks_demo/default/files_data/transaction_data/")  # Source directory path
    )

#### **_[Order Items Raw](url)_**

In [0]:
# Import Spark Declarative Pipelines module
from pyspark import pipelines as dp

# Define a streaming table in the bronze layer for order items data
@dp.table(
    name="databricks_demo.bronze.tb_order_items_raw",  # Fully qualified table name in Unity Catalog
    comment="The order items table. ingested from order_items.csv file"  # Table description
)
def order_items_raw():
    # Configure Auto Loader to incrementally ingest CSV files
    return (
        spark.readStream.format("cloudFiles")  # Use Auto Loader for incremental file ingestion
        .option("cloudFiles.format", "csv")  # Specify source file format as CSV
        .option("cloudFiles.inferColumnTypes", "true")  # Automatically infer column data types
        .option("pathGlobFilter", "order_items.csv")  # Filter to only process order_items.csv file
        .load("/Volumes/databricks_demo/default/files_data/transaction_data/")  # Source directory path
    )

## **_[Silver Layer](url)_**

#### **_[Customer Data Dim](url)_**

In [0]:
# Import Spark Declarative Pipelines module
from pyspark import pipelines as dp

# Define a streaming table in the silver layer with data quality expectations
@dp.table(
    name="databricks_demo.silver.tb_customers_dim",  # Fully qualified table name in Unity Catalog
    comment="The customers table, created from the raw customers table of the bronze schema"  # Table description
)
@dp.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL")  # Drop rows with NULL customer_id
@dp.expect_or_fail("valid_first_name", "first_name IS NOT NULL")  # Fail pipeline if first_name is NULL
@dp.expect_or_fail("valid_last_name", "last_name IS NOT NULL")  # Fail pipeline if last_name is NULL
def customers_clean():
    # Read streaming data from bronze customers table
    raw = spark.readStream.table("databricks_demo.bronze.tb_customers_raw")
    
    # Transform and select columns for silver layer
    return raw.select(
        "customer_id",  # Keep customer ID
        "first_name",  # Keep first name
        "last_name",  # Keep last name
        (raw.first_name + " " + raw.last_name).alias("full_name"),  # Create full name from first + last
        "email",  # Keep email
        "phone",  # Keep phone
        "address",  # Keep address
        "city",  # Keep city
        "state",  # Keep state
        "postal_code",  # Keep postal code
        "country",  # Keep country
        raw.signup_date.cast("date").alias("signup_date"),  # Convert signup_date to date type
        "customer_segment",  # Keep customer segment
        "is_active"  # Keep active status
    )

#### **_[Product Data Dim](url)_**

In [0]:
# Import Spark Declarative Pipelines module
from pyspark import pipelines as dp

# Define a streaming table in the silver layer with data quality expectations
@dp.table(
    name="databricks_demo.silver.tb_procuducts_dim",  # Fully qualified table name in Unity Catalog
    comment="The products table, created from the raw products table of the bronze schema"  # Table description
)
@dp.expect_or_drop("valid_product_id", "product_id IS NOT NULL")  # Drop rows with NULL product_id
def products_clean():
    # Read streaming data from bronze products table
    raw = spark.readStream.table("databricks_demo.bronze.tb_products_raw")
    
    # Transform and select columns for silver layer
    return raw.select(
        "product_id",  # Keep product ID
        "product_name",  # Keep product name
        "category",  # Keep category
        "unit_price",  # Keep unit price
        "cost_price",  # Keep cost price
        "stock_quantity",  # Keep stock quantity
        "reorder_level",  # Keep reorder level
        "supplier_id",  # Keep supplier ID
        "product_description",  # Keep product description
        "date_added",  # Keep date added
        "is_active"  # Keep active status
    )

#### **_[Orders Data Clean](url)_**

In [0]:
# Import necessary modules
from pyspark import pipelines as dp
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Define a materialized view for payment dimension
@dp.materialized_view(
    name="databricks_demo.silver.vw_payment_dim",
    comment="The payment methods dimension table, created from the orders table"
)
def payment_dim():
    # Step 1: Read batch (not streaming) from bronze orders table
    raw = spark.read.table("databricks_demo.bronze.tb_orders_raw")
    # Step 2: Deduplicate payment_methods
    payment_methods = raw.select("payment_method").dropDuplicates(["payment_method"])
    # Step 3: Assign sequential payment_method_id (serial_number)
    window = Window.orderBy("payment_method")
    payment_dim_df = payment_methods.withColumn(
        "payment_method_id",
        row_number().over(window)
    )
    # Step 4: Return only the payment_method_id and payment_method columns
    return payment_dim_df.select("payment_method_id", "payment_method")

#### **_[Calendar Table Dim](url)_**

In [0]:
# Import necessary modules
from pyspark import pipelines as dp
from pyspark.sql.functions import col, year, month, dayofmonth, weekofyear, dayofweek, quarter, date_format

# Define a materialized view for calendar dimension
@dp.materialized_view(
    name="databricks_demo.silver.vw_calendar_dim",
    comment="Calendar dimension table based on distinct order_date values from orders raw table"
)
def calendar_dim():
    # Step 1: Read batch data from orders table
    orders = spark.read.table("databricks_demo.bronze.tb_orders_raw")
    # Step 2: Select unique order_date values
    calendar_dates = orders.select("order_date").dropDuplicates(["order_date"])
    # Step 3: Add calendar-related fields, datekey as yyyymmdd string
    calendar_df = calendar_dates.select(
        date_format(col("order_date"), "yyyyMMdd").alias("datekey"),  # DateKey in yyyymmdd format
        col("order_date"),
        year("order_date").alias("year"),
        month("order_date").alias("month"),
        dayofmonth("order_date").alias("day"),
        weekofyear("order_date").alias("week"),
        dayofweek("order_date").alias("weekday"),
        quarter("order_date").alias("quarter")
    )
    # Step 4: Return completed calendar DataFrame
    return calendar_df

#### **_[Order Status Dim](url)_**

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, lit, row_number
from pyspark.sql.window import Window

@dp.materialized_view(
    name="databricks_demo.silver.vw_order_status_dim",
    comment="This is a materialized view of the order status dimension table"
)
def order_status_dim():
    # Step 1: Read batch data from orders table
    orders = spark.read.table("databricks_demo.bronze.tb_orders_raw")  # Read orders raw table in batch mode
    # Step 2: Select unique order_status values
    statuses = orders.select("order_status").dropDuplicates(["order_status"])  # Deduplicate status values
    # Step 3: Assign sequential order_status_id using row_number over sorted order_status
    window = Window.orderBy("order_status")  # Define window for sorting
    statuses = statuses.withColumn("order_status_id", row_number().over(window))  # Add incremental ID
    # Step 4: Select final columns for dimension
    order_status_df = statuses.select("order_status_id", "order_status")  # Prepare final output
    # Step 5: Return completed DataFrame
    return order_status_df

#### **_[Shipping Address Dim](url)_**

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

@dp.materialized_view(
    name="databricks_demo.silver.vw_order_shipping_address_dim",
    comment="This is a materialized view of the order shipping address dimension table"
)
def order_shipping_address_dim():
    # Step 1: Read batch data from orders table
    orders = spark.read.table("databricks_demo.bronze.tb_orders_raw")
    # Step 2: Select unique shipping addresses
    shipping_addresses = orders.select("customer_id", "order_id", "shipping_address").distinct()
    # Step 3: Create a window function for sorting
    window_spec = Window.orderBy(col("customer_id").asc(), col("order_id").asc(), col("shipping_address").asc())
    # Step 4: Generate unique shipping_address_id
    shipping_addresses = shipping_addresses.withColumn("shipping_address_id", row_number().over(window_spec))
    # Step 5: Select final columns for dimension
    order_shipping_address_df = shipping_addresses.select("shipping_address_id", "customer_id", "order_id", "shipping_address")
    # Step 6: Return completed DataFrame
    return order_shipping_address_df


#### **_[Orders Fact Table](url)_**

In [0]:
# Import necessary modules
from pyspark import pipelines as dp

# Define a streaming table for orders fact
@dp.table(
    name="databricks_demo.silver.tb_orders_fact",
    comment="The orders fact table, created from the raw orders table of the bronze schema"
)
@dp.expect_or_drop("valid_order_id", "order_id IS NOT NULL")  # Drop rows with NULL order_id
def orders_fact():
    # Step 1: Read streaming data from bronze orders table
    orders = spark.readStream.table("databricks_demo.bronze.tb_orders_raw")
    # Step 2: Create temporary view for orders
    orders.createOrReplaceTempView("orders_temp")
    
    # Step 3: Use SQL to join all dimensions with explicit table aliases
    result = spark.sql("""
        SELECT 
            ord.order_id,
            ord.customer_id,
            cal.datekey,
            status.order_status_id,
            ord.total_amount,
            ord.discount_percent,
            ord.tax_amount,
            pay.payment_method_id,
            ship.shipping_address_id,
            ord.shipping_date,
            ord.delivery_date
        FROM orders_temp ord
        LEFT JOIN databricks_demo.silver.vw_calendar_dim cal
            ON ord.order_date = cal.order_date
        LEFT JOIN databricks_demo.silver.vw_order_status_dim status
            ON ord.order_status = status.order_status
        LEFT JOIN databricks_demo.silver.vw_payment_dim pay
            ON ord.payment_method = pay.payment_method
        LEFT JOIN databricks_demo.silver.vw_order_shipping_address_dim ship
            ON ord.customer_id = ship.customer_id
            AND ord.order_id = ship.order_id
            AND ord.shipping_address = ship.shipping_address
    """)
    # Step 4: Return the enriched streaming DataFrame    
    return result

# **_[Gold Layer](url)_**

#### **_[Order Fact](url)_**

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col

# Define a feature table for orders
@dp.table(
    name="databricks_demo.gold.vw_orders_fact_dtl",
    comment="The orders feature table, created from the orders fact table of the silver schema"
)
def orders_feature_table():
    # Step 1: Read streaming data from orders fact table
    orders = spark.readStream.table("databricks_demo.silver.tb_orders_fact")
    # Step 2: Create temporary view for the streaming orders DataFrame
    orders.createOrReplaceTempView("orders_temp")
    # Step 3: Use SQL to select features from the temp view
    features = spark.sql("""
        SELECT 
            order_id,
            customer_id,
            datekey,
            order_status_id,
            total_amount,
            discount_percent,
            tax_amount,
            payment_method_id,
            shipping_address_id,
            shipping_date,
            delivery_date
        FROM orders_temp
    """)
    # Step 4: Return the features DataFrame
    return features


In [0]:
from pyspark import pipelines as dp

# Define a materialized view or temporary view for gold order items
@dp.temporary_view(
    name="gold_order_items_view",
    comment="Pipeline-private view for order items in the gold layer"
)
def gold_order_items_view():
    df = spark.read.table("databricks_demo.bronze.tb_order_items_raw")
    return df.select(
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "discount_percent",
        "unit_price"
    )